## Dataset 5: Weather Sensors Data Profiling & Rescue Notebook

In [41]:
import os
import re
import pandas as pd
import numpy as np

In [42]:
# Load Raw Weather Sensors Dataset
raw_excel_path = "../data/raw/track3_weather_sensors.xlsx"
if not os.path.exists(raw_excel_path):
    raw_excel_path = "track3_weather_sensors.xlsx"
df_raw = pd.read_excel(raw_excel_path)

In [43]:
print("Dataset Shape:", df_raw.shape)


Dataset Shape: (15000, 7)


In [44]:

print("First 10 Rows of Raw Weather Dataset:")
df_raw.head(10)

First 10 Rows of Raw Weather Dataset:


,sensor_id,timestamp,temperature,temp_unit,rainfall,rain_unit,humidity_percent
0,SEN047,28/08/2026,68,f,43.80,mm,46.0
1,SEN034,2026-07-03 07:08:18 IST,85.3,°F,1.88,in,55.0
2,SEN020,NaN,18.9,Celsius,19.90,MM,NaN
3,SEN035,2026-07-09 11:08:41 UTC,71.8,Fahrenheit,0.91,inch,49.0
4,SEN048,2026-06-09 13:06:26 IST,39.8°C,NaN,43.90,mm,81.0
5,SEN035,17/03/2026,21.8°C,NaN,28.90,MM,47.0
6,SEN013,07-01-2026 05:48 PM,75,f,32.40,mm,43.0
7,SEN040,2026-06-12 09:52:51 IST,92.5,°F,NaN,NaN,92.0
8,SEN027,2026-06-02 16:00:04 UTC,22.1,°C,-48.10,mm,36.0
9,SEN034,NaN,30.3°C,NaN,NaN,NaN,51.0


### Step 1: Raw Excel Data Profiling and Messiness Audit
**Problem Strategy**:
Before applying rescue transformations, we audit raw data messiness across:
1. `temperature` & `temp_unit`: Identify Fahrenheit vs Celsius unit variations (6,020 Fahrenheit records) and 2,263 missing units.
2. `rainfall` & `rain_unit`: Detect inches vs millimeters unit variations (3,796 inches records) and 1,518 negative rainfall sign anomalies.
3. `timestamp`: Scan for UTC vs IST timezone tags (5,141 UTC timestamps requiring +5:30 IST offset conversion) and 1,555 missing timestamps.
4. `humidity_percent`: Inspect missing values (1,500 missing) and relative humidity distributions.

In [45]:
# Raw Weather Sensors Data Profiling Check
print("Raw Dataset Dimensions Scan:\n")
print("Total Raw Sensor Rows:", len(df_raw))
print("Unique Sensor IDs count:", df_raw['sensor_id'].nunique())

Raw Dataset Dimensions Scan:

Total Raw Sensor Rows: 15000
Unique Sensor IDs count: 51


In [46]:
print("Temperature Unit Distribution Scan:")
print(df_raw['temp_unit'].value_counts(dropna=False))

Temperature Unit Distribution Scan:
temp_unit
NaN           2263
c             1733
°C            1683
C             1679
Celsius       1622
f             1551
°F            1539
F             1466
Fahrenheit    1464
Name: count, dtype: int64


In [47]:
print("Rainfall Unit & Negative Anomaly Scan:")
print("Rainfall unit value counts:")
print(df_raw['rain_unit'].value_counts(dropna=False))

Rainfall Unit & Negative Anomaly Scan:
Rainfall unit value counts:
rain_unit
mm             4459
MM             3038
millimeters    2916
inches         1285
in             1270
inch           1241
NaN             791
Name: count, dtype: int64


In [48]:
# Parse numeric values safely, returning NaN for invalid entries
def parse_numeric(val):
    try:
        return float(val)
    except:
        return np.nan

In [49]:
raw_rain = df_raw['rainfall'].apply(parse_numeric)

In [50]:
print("Negative rainfall count:", (raw_rain < 0).sum())

Negative rainfall count: 1518


In [51]:
print("Timestamp Timezone Scan:")
utc_cnt = df_raw['timestamp'].astype(str).str.contains('UTC', case=False).sum()
ist_cnt = df_raw['timestamp'].astype(str).str.contains('IST', case=False).sum()
print("UTC timestamp count:", utc_cnt)
print("IST timestamp count:", ist_cnt)
print("Missing timestamp count:", df_raw['timestamp'].isnull().sum())

Timestamp Timezone Scan:
UTC timestamp count: 5141
IST timestamp count: 6038
Missing timestamp count: 1555


In [52]:
print("Humidity Missing Count:", df_raw['humidity_percent'].isnull().sum())

Humidity Missing Count: 1500


### Step 2: Temperature Unit Standardization & Fahrenheit to Celsius Conversion

**Problem Strategy**:
1. Temperature Unit Parsing: Raw temperature readings are recorded in mixed units (`C`, `°C`, `Celsius`, `F`, `°F`, `Fahrenheit`).
2. Unit Standardisation: Convert all Fahrenheit readings to Celsius using the formula `Celsius = (F - 32) * 5/9` (6,020 Fahrenheit records converted).
3. Missing Unit Inference: For 2,263 missing temperature unit entries (`NaN`), infer unit based on raw value range (if value > 50 -> Fahrenheit, else Celsius). Standardize all units to `Celsius`.


In [53]:
# Check Raw Temperature Units & Summary Stats
print("Raw Temperature Unit Distribution:")
print(df_raw['temp_unit'].value_counts(dropna=False))


Raw Temperature Unit Distribution:
temp_unit
NaN           2263
c             1733
°C            1683
C             1679
Celsius       1622
f             1551
°F            1539
F             1466
Fahrenheit    1464
Name: count, dtype: int64


In [54]:
# Parse function to extract numeric value of temperature
def parse_temp(val):
    try:
        s = str(val).replace('°C', '').replace('°F', '').replace('C', '').replace('F', '').strip()
        return float(s)
    except:
        return np.nan


In [55]:
df = df_raw.copy()
df['raw_temp_num'] = df['temperature'].apply(parse_temp)

In [56]:
# Function to determine if a temperature reading is in Fahrenheit
def is_fahrenheit(row):
    unit = str(row['temp_unit']).lower().strip()
    if unit in ['f', '°f', 'fahrenheit']:
        return True
    if pd.isna(row['temp_unit']) or unit == 'nan':
        if row['raw_temp_num'] > 50.0:
            return True
    return False


In [57]:
df['is_fahrenheit_flag'] = df.apply(is_fahrenheit, axis=1)


In [58]:
# Convert Fahrenheit to Celsius: (F - 32) * 5/9
df['clean_temperature_c'] = np.where(
    df['is_fahrenheit_flag'],
    (df['raw_temp_num'] - 32.0) * (5.0 / 9.0),
    df['raw_temp_num']
).round(2)

In [59]:
df['clean_temp_unit'] = 'Celsius'

 #### BEFORE vs AFTER comparison

In [60]:
print("Before vs After Temperature Conversion Verification:")
print("Total Fahrenheit records converted to Celsius:", df['is_fahrenheit_flag'].sum())
print("Raw temperature range: Min =", df['raw_temp_num'].min(), ", Max =", df['raw_temp_num'].max(), ", Mean =", round(df['raw_temp_num'].mean(), 2))
print("Rescued Celsius range: Min =", df['clean_temperature_c'].min(), ", Max =", df['clean_temperature_c'].max(), ", Mean =", round(df['clean_temperature_c'].mean(), 2))

Before vs After Temperature Conversion Verification:
Total Fahrenheit records converted to Celsius: 6020
Raw temperature range: Min = 15.0 , Max = 104.0 , Mean = 49.07
Rescued Celsius range: Min = 15.0 , Max = 40.0 , Mean = 27.43


In [62]:
print("Sample Fahrenheit -> Celsius Conversions:")
print(df[df['is_fahrenheit_flag']][['temperature', 'temp_unit', 'clean_temperature_c']].head(10))

Sample Fahrenheit -> Celsius Conversions:
   temperature   temp_unit  clean_temperature_c
0           68           f                20.00
1         85.3          °F                29.61
3         71.8  Fahrenheit                22.11
6           75           f                23.89
7         92.5          °F                33.61
11       102.2           F                39.00
13        66.9           f                19.39
14        73.2           F                22.89
15       102.4  Fahrenheit                39.11
16        67.6           f                19.78


### Step 3: Rainfall Unit Standardization (Inches -> MM) & Negative Anomaly Correction

**Problem Strategy**:
1. Negative Anomaly Correction: 1,518 rainfall entries contain negative sign logging errors (e.g., `-50.0`). We fix them using `abs()` and create `is_negative_rainfall_anomaly` flag.
2. Inches to MM Conversion: 3,796 rainfall records are recorded in `inches` (`in`, `inch`, `inches`). We convert them to millimeters (`mm = inches * 25.4`).
3. Unit Standardisation: Standardize `rain_unit` to `mm` across all records.


## Before Check Raw Rainfall Units & Negative Values Scan


In [64]:
print("Raw Rainfall Unit Distribution:\n")
print(df_raw['rain_unit'].value_counts(dropna=False))
def parse_numeric(val):
    try:
        return float(val)
    except:
        return np.nan
raw_rain = df_raw['rainfall'].apply(parse_numeric)
print("\nNegative rainfall entries count before rescue:", (raw_rain < 0).sum())


Raw Rainfall Unit Distribution:

rain_unit
mm             4459
MM             3038
millimeters    2916
inches         1285
in             1270
inch           1241
NaN             791
Name: count, dtype: int64

Negative rainfall entries count before rescue: 1518


In [65]:
# Parse raw rainfall to float
df['raw_rain_num'] = df['rainfall'].apply(parse_numeric) 

In [66]:
# Fix negative sign anomalies using abs()
df['clean_rainfall_mm'] = df['raw_rain_num'].abs()
df['is_negative_rainfall_anomaly'] = np.where(df['raw_rain_num'] < 0, 1, 0)

In [67]:
# Identify inches units and convert to MM 
def is_inches(row):
    unit = str(row['rain_unit']).lower().strip()
    if unit in ['in', 'inch', 'inches']:
        return True
    return False


In [68]:
df['is_inches_flag'] = df.apply(is_inches, axis=1)

In [70]:
df['clean_rainfall_mm'] = np.where(
    df['is_inches_flag'],
    df['clean_rainfall_mm'] * 25.4,
    df['clean_rainfall_mm']
).round(2)

In [ ]:
df['clean_rain_unit'] = 'mm'

### BEFORE vs AFTER comparison

In [73]:
print("Before vs After Rainfall Rescue Verification:")
print()
print("Negative rainfall entries corrected count:", df['is_negative_rainfall_anomaly'].sum())
print("Inches rainfall records converted to MM count:", df['is_inches_flag'].sum())
print("Raw total rainfall sum:", round(raw_rain.sum(), 2), "| Rescued total MM rainfall sum:", round(df['clean_rainfall_mm'].sum(), 2))


Before vs After Rainfall Rescue Verification:

Negative rainfall entries corrected count: 1518
Inches rainfall records converted to MM count: 3796
Raw total rainfall sum: 187220.67 | Rescued total MM rainfall sum: 355722.41


In [74]:
print("Sample Inches to MM Conversions:")
df[df['is_inches_flag']][['rainfall', 'rain_unit', 'clean_rainfall_mm']].head(10)

Sample Inches to MM Conversions:


,rainfall,rain_unit,clean_rainfall_mm
1,1.88,in,47.75
3,0.91,inch,23.11
23,0.23,inches,5.84
26,0.78,in,19.81
35,1.84,in,46.74
36,1.58,inch,40.13
37,0.67,inch,17.02
38,0.57,inch,14.48
40,1.28,in,32.51
43,1.70,inch,43.18
